# Laboratorio: reconstrucción de frontend desde capturas

Este laboratorio sigue la aplicación real FastAPI + Streamlit + MongoDB + proveedor OpenAI Vision. El prototipo original era un clasificador demostrativo con datos sintéticos; no generaba interfaces. Aquí no se entrena ningún modelo. Una captura no contiene el código fuente original y el resultado puede diferir en fuentes, interacciones, diseño responsive y contenido ausente.

## 1. Objetivo, alcance y arquitectura

Streamlit carga captura o URL pública; FastAPI valida, llama al modelo de visión y guarda el HTML/CSS/JavaScript en MongoDB. La vista previa se aísla en un iframe sandbox y el servidor nunca ejecuta el código generado. URLs privadas, páginas con login, contenido dinámico y bloqueo de automatización tienen limitaciones.

## 2. Preparación del entorno Conda en macOS

Ejecuta en Anaconda Terminal desde la raíz del proyecto:

```bash
conda create -n screenshot-ui python=3.11 -y
conda activate screenshot-ui
python -m pip install -r requirements.txt
python -m playwright install chromium
```

In [ ]:
from pathlib import Path
import os, sys, requests, json
ROOT = Path.cwd().resolve()
if not (ROOT / 'backend' / 'api.py').exists():
    raise RuntimeError('Abre este notebook desde la raíz del proyecto.')
sys.path.insert(0, str(ROOT))
print('Raíz:', ROOT)

## 3. Configuración de `.env` sin mostrar secretos

Copia `.env.example` a `.env` y configura `OPENAI_API_KEY`, `OPENAI_MODEL`, `MONGO_URI`, `MONGO_DB` y `API_URL`. Nunca imprimas ni guardes la clave en una salida del notebook.

In [ ]:
from dotenv import load_dotenv
load_dotenv(ROOT / '.env', override=True)
required = ['OPENAI_API_KEY', 'MONGO_URI', 'MONGO_DB', 'API_URL']
status = {name: bool(os.getenv(name)) for name in required}
print('Variables configuradas (sin valores):', status)
if not all(status.values()): print('Falta configuración: edita .env y vuelve a ejecutar esta celda.')

## 4. Verificación de dependencias y conexión a MongoDB

MongoDB crea la base y colección tras la primera inserción. En Compass conecta al URI local configurado y revisa `web_screenshot_builder.projects`.

In [ ]:
from importlib.util import find_spec
for module in ['fastapi','streamlit','pymongo','PIL','openai','playwright']:
    print(module, 'OK' if find_spec(module) else 'FALTA')
try:
    from pymongo import MongoClient
    client = MongoClient(os.getenv('MONGO_URI','mongodb://127.0.0.1:27017'), serverSelectionTimeoutMS=2500)
    client.admin.command('ping')
    print('MongoDB conectado; base:', os.getenv('MONGO_DB','web_screenshot_builder'))
except Exception as exc:
    print('MongoDB no disponible. Inicia MongoDB y verifica MONGO_URI en .env. (' + type(exc).__name__ + ')')

## 5. Carga de captura o captura desde URL

Elige una ruta. La API captura URLs públicas con Playwright Chromium; no resuelve páginas que requieren sesión y puede encontrar bloqueo o contenido incompleto.

In [ ]:
from getpass import getpass
from IPython.display import display, Image as DisplayImage
source_mode = input('Escribe archivo o url: ').strip().lower()
image_path = None
source_url = None
if source_mode == 'archivo':
    raw = input('Ruta de imagen JPG/PNG/WEBP: ').strip()
    image_path = Path(raw).expanduser()
    if not image_path.is_file(): print('No existe el archivo; selecciona una captura real.')
elif source_mode == 'url':
    source_url = input('URL pública http(s): ').strip()
else: print('Selecciona archivo o url y ejecuta la celda de nuevo.')

## 6. Validación y visualización de la imagen

El backend valida contenido real, tipo, tamaño máximo de 10 MB y 25 megapíxeles. Esta celda valida la carga local; en URL la API aplica la misma validación sobre la captura.

In [ ]:
from PIL import Image
image_bytes = None
if image_path and image_path.is_file():
    image_bytes = image_path.read_bytes()
    try:
        image = Image.open(image_path); image.verify(); image = Image.open(image_path)
        if image.format not in ('PNG','JPEG','WEBP') or len(image_bytes)>10*1024*1024 or image.width*image.height>25_000_000: raise ValueError('Formato o tamaño fuera de límites')
        print('Imagen válida:', image.format, image.size, f'{len(image_bytes)/1024/1024:.2f} MB')
        display(DisplayImage(data=image_bytes))
    except Exception as exc:
        image_bytes = None; print('Imagen no válida:', type(exc).__name__, str(exc))
elif not source_url: print('Carga una imagen o indica una URL antes de continuar.')

## 7. Llamada al backend y generación del frontend

Inicia API (`uvicorn backend.api:app --reload --port 8000`) y MongoDB en terminales separadas. Se llama al backend real; no se muestra contenido de ejemplo como si fuera una generación. Requiere clave de proveedor válida.

In [ ]:
API = os.getenv('API_URL','http://127.0.0.1:8000').rstrip('/')
project = None
try:
    if image_bytes:
        name = image_path.name
        response = requests.post(API+'/projects', files={'file':(name,image_bytes,'image/png' if name.lower().endswith('.png') else 'image/jpeg')}, timeout=120)
    elif source_url:
        response = requests.post(API+'/projects/from-url', json={'url':source_url}, timeout=150)
    else: response = None; print('Falta una captura válida.')
    if response is not None:
        if response.ok: project=response.json(); print('Proyecto guardado:', project['id'], project['provider'], project['model'])
        else: print('Backend:', response.status_code, response.json().get('detail','Error sin detalle'))
except requests.RequestException as exc: print('No se pudo conectar al backend. Inícialo y verifica API_URL. ('+type(exc).__name__+')')

## 8. Visualización y descarga del resultado

Descarga ZIP con HTML/CSS/JavaScript. No se ejecuta código generado dentro del notebook.

In [ ]:
from IPython.display import display, HTML
if project:
    display(HTML('<h3>HTML</h3><pre>'+project['html'].replace('&','&amp;').replace('<','&lt;')+'</pre>'))
    print('CSS:\n', project['css'][:12000])
    print('JavaScript:\n', project.get('javascript','')[:12000])
    import zipfile
    output = ROOT / 'reconstruccion_frontend.zip'
    with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED) as z:
        z.writestr('index.html','<!doctype html><html><head><meta charset="utf-8"><link rel="stylesheet" href="styles.css"></head><body>'+project['html']+'<script src="script.js"></script></body></html>')
        z.writestr('styles.css',project['css']); z.writestr('script.js',project.get('javascript',''))
    print('Descarga lista:',output.name)
else: print('No hay resultado: revisa imagen, clave, API y MongoDB.')

## 9. Consulta y eliminación de proyectos guardados

Consulta desde la API; borrar requiere confirmar explícitamente el ID. También puedes gestionar proyectos en Streamlit o Compass.

In [ ]:
saved=[]
try:
    r=requests.get(API+'/projects',timeout=10); r.raise_for_status(); saved=r.json(); print(json.dumps(saved,ensure_ascii=False,indent=2))
except Exception as exc: print('No se pudo consultar: inicia API/MongoDB. ('+type(exc).__name__+')')
if saved:
    delete_id=input('Para borrar un proyecto, pega su ID; Enter conserva todos: ').strip()
    if delete_id:
        try:
            r=requests.delete(API+'/projects/'+delete_id,timeout=10); r.raise_for_status(); print('Eliminado:',delete_id)
        except Exception as exc: print('No se pudo eliminar:',type(exc).__name__)

## 10. Evaluación de similitud visual y limitaciones

Compara la captura original con la vista previa aislada en la aplicación. Evalúa estructura, color, espaciado, tipografía y respuesta a distintos anchos. Una métrica automática de píxeles puede penalizar antialiasing y variaciones de fuentes; una puntuación no implica fidelidad semántica. URL dinámica/login/bloqueos, contenido no visible, fuentes propietarias e interacciones originales limitan la reconstrucción. No existe código fuente original recuperable desde la imagen.

## 11. Pruebas y conclusiones

Desde Anaconda Terminal: `python -m pytest -q`. Las pruebas cubren validación y persistencia simulada sin llamadas pagadas ni MongoDB. El proveedor real requiere credencial propia y acceso al modelo configurado. La aplicación no entrena modelos; la salida es código generado que solo se previsualiza en iframe sandbox.